# Week 2 - Convolution and Blur

This notebook accompanies the second MATH 435 slide deck.

## Goal

By the end, you should be able to:

- compute a simple 1D convolution;
- interpret a blur kernel as a point spread function;
- apply 2D convolution to a real image;
- compare boundary conditions;
- see why naive deblurring amplifies noise.

## Setup

The notebook uses NumPy, SciPy, scikit-image, and Plotly.

If you run this outside Google Colab and a package is missing, install the course requirements from the repository root:

```bash
python3 -m pip install -r requirements.txt
```

In [ ]:
import numpy as np
from scipy import ndimage, signal
from skimage import data
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def gaussian_kernel(size, sigma):
    axis = np.arange(-(size // 2), size // 2 + 1)
    xx, yy = np.meshgrid(axis, axis)
    kernel = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    return kernel / kernel.sum()


def motion_kernel(size):
    kernel = np.zeros((size, size), dtype=float)
    kernel[size // 2, :] = 1.0
    return kernel / kernel.sum()


def show_heatmaps(images, titles, colorscale="Gray", zmin=None, zmax=None, height=430):
    fig = make_subplots(rows=1, cols=len(images), subplot_titles=titles)
    for index, image in enumerate(images, start=1):
        fig.add_trace(
            go.Heatmap(
                z=image,
                colorscale=colorscale,
                zmin=zmin,
                zmax=zmax,
                showscale=index == len(images),
            ),
            row=1,
            col=index,
        )
        fig.update_xaxes(showticklabels=False, row=1, col=index)
        fig.update_yaxes(autorange="reversed", showticklabels=False, row=1, col=index)
    fig.update_layout(height=height, margin=dict(l=20, r=20, t=60, b=20))
    fig.show()


def show_signal_panels(series, titles, height=360):
    fig = make_subplots(rows=1, cols=len(series), subplot_titles=titles)
    for index, values in enumerate(series, start=1):
        x_values = np.arange(len(values))
        fig.add_trace(go.Bar(x=x_values, y=values), row=1, col=index)
        fig.update_xaxes(title_text="index", row=1, col=index)
    fig.update_layout(height=height, showlegend=False, margin=dict(l=20, r=20, t=60, b=45))
    fig.show()

## Step 1 - 1D Convolution by Hand

Start with a short signal and a three-tap smoothing kernel.

In [ ]:
x = np.array([0, 0, 1, 3, 2, 0, 0], dtype=float)
h = np.array([0.25, 0.5, 0.25])
y = np.convolve(x, h, mode="same")

print("x:", x)
print("h:", h)
print("y:", y)

show_signal_panels(
    [x, h, y],
    ["signal x", "kernel h", "same-size convolution y"],
)

### Exercise 1

Compute the central output value by hand, then compare with Python.

In [ ]:
central_index = 3
by_hand = 0.25 * x[central_index - 1] + 0.5 * x[central_index] + 0.25 * x[central_index + 1]

print("by hand:", by_hand)
print("from convolution:", y[central_index])

## Step 2 - Blur Kernels as Point Spread Functions

A point spread function describes how one ideal bright point is spread by the imaging system.

In [ ]:
identity = np.zeros((7, 7))
identity[3, 3] = 1.0

box = np.ones((7, 7)) / 49
gaussian = gaussian_kernel(7, 1.2)
motion = motion_kernel(7)

kernels = [identity, box, gaussian, motion]
titles = ["identity", "box average", "gaussian", "motion"]

show_heatmaps(kernels, titles, colorscale="Viridis", zmin=0, height=360)

for title, kernel in zip(titles, kernels):
    print(title, "sum =", round(float(kernel.sum()), 6))

### Exercise 2

Build a new kernel. Keep the sum equal to 1 if you want average brightness to be preserved.

In [ ]:
# TODO: change these values and rerun.
kernel_size = 9
sigma = 2.0

student_kernel = gaussian_kernel(kernel_size, sigma)

print("shape:", student_kernel.shape)
print("sum:", student_kernel.sum())
show_heatmaps([student_kernel], ["student gaussian kernel"], colorscale="Viridis", zmin=0, height=420)

## Step 3 - Blur a Real Image

A fixed blur kernel gives a linear imaging operator.

In [ ]:
image = data.camera().astype(float) / 255.0
image = image[80:400, 90:410]

box13 = np.ones((13, 13)) / (13 * 13)
gaussian21 = gaussian_kernel(21, 3.0)
motion25 = motion_kernel(25)

box_blur = signal.convolve2d(image, box13, mode="same", boundary="symm")
gaussian_blur = signal.convolve2d(image, gaussian21, mode="same", boundary="symm")
motion_blur = signal.convolve2d(image, motion25, mode="same", boundary="symm")

show_heatmaps(
    [image, box_blur, gaussian_blur, motion_blur],
    ["original", "box blur", "gaussian blur", "motion blur"],
    zmin=0,
    zmax=1,
    height=430,
)

### Exercise 3

Change the blur parameters. Which details disappear first?

In [ ]:
# TODO: change kernel_size and sigma.
kernel_size = 31
sigma = 5.0

experimental_kernel = gaussian_kernel(kernel_size, sigma)
experimental_blur = signal.convolve2d(image, experimental_kernel, mode="same", boundary="symm")

show_heatmaps(
    [image, experimental_blur],
    ["original", f"gaussian blur: size={kernel_size}, sigma={sigma}"],
    zmin=0,
    zmax=1,
    height=460,
)

## Step 4 - Boundary Conditions

Near the image boundary, the kernel asks for pixels outside the image. Boundary conditions decide what those pixels mean.

In [ ]:
boundary_crop = data.camera().astype(float)[90:218, 315:443] / 255.0
boundary_kernel = np.ones((17, 17)) / (17 * 17)

zero_padding = ndimage.convolve(boundary_crop, boundary_kernel, mode="constant", cval=0.0)
reflect = ndimage.convolve(boundary_crop, boundary_kernel, mode="reflect")
wrap = ndimage.convolve(boundary_crop, boundary_kernel, mode="wrap")

show_heatmaps(
    [boundary_crop, zero_padding, reflect, wrap],
    ["original crop", "zero padding", "reflect", "wrap"],
    zmin=0,
    zmax=1,
    height=430,
)

### Exercise 4

Change `mode` to `"nearest"` or `"mirror"` in the cell below. Compare it with `"constant"`, `"reflect"`, and `"wrap"`.

In [ ]:
# TODO: try "nearest", "mirror", "constant", "reflect", or "wrap".
boundary_mode = "nearest"

boundary_result = ndimage.convolve(boundary_crop, boundary_kernel, mode=boundary_mode, cval=0.0)

show_heatmaps(
    [boundary_crop, boundary_result],
    ["original crop", f"mode={boundary_mode}"],
    zmin=0,
    zmax=1,
    height=460,
)

## Step 5 - Naive Deblurring Is Unstable

A blur removes or attenuates high-frequency information. Trying to invert that attenuation can amplify noise.

In [ ]:
n = 256
t = np.linspace(0, 1, n, endpoint=False)

clean = np.sin(2 * np.pi * 6 * t) + 0.45 * np.sin(2 * np.pi * 34 * t)
clean = clean / np.max(np.abs(clean))

axis = np.arange(-18, 19)
blur_kernel = np.exp(-(axis**2) / (2 * 3.0**2))
blur_kernel = blur_kernel / blur_kernel.sum()

blurred = np.convolve(clean, blur_kernel, mode="same")

rng = np.random.default_rng(7)
noise_level = 0.035
noisy = blurred + noise_level * rng.standard_normal(n)

padded_kernel = np.zeros(n)
padded_kernel[: len(blur_kernel)] = blur_kernel
padded_kernel = np.roll(padded_kernel, -(len(blur_kernel) // 2))

H = np.fft.fft(padded_kernel)
stabilizer = 1e-3
naive_inverse = np.real(np.fft.ifft(np.fft.fft(noisy) / (H + stabilizer)))
naive_inverse = naive_inverse / max(1.0, np.max(np.abs(naive_inverse)))

fig = make_subplots(
    rows=1,
    cols=4,
    subplot_titles=["clean", "blurred", "blurred + noise", "naive inverse"],
)
for index, values in enumerate([clean, blurred, noisy, naive_inverse], start=1):
    fig.add_trace(go.Scatter(x=t, y=values, mode="lines"), row=1, col=index)
    fig.update_yaxes(range=[-1.25, 1.25], row=1, col=index)
fig.update_layout(height=360, showlegend=False, margin=dict(l=20, r=20, t=60, b=35))
fig.show()

### Exercise 5

The stabilizer limits noise amplification. Try changing it by powers of ten.

In [ ]:
# TODO: add or remove stabilizer values.
stabilizers = [1e-1, 1e-2, 1e-3]

for value in stabilizers:
    estimate = np.real(np.fft.ifft(np.fft.fft(noisy) / (H + value)))
    estimate = estimate / max(1.0, np.max(np.abs(estimate)))
    rms_error = np.sqrt(np.mean((estimate - clean) ** 2))
    print(f"stabilizer={value:g}, RMS error={rms_error:.3f}")

### Coherence Check - Blur as a Forward Model

        - **Operator.** Which blur kernel and boundary condition define the forward operator?
- **Assumption.** When is a spatially invariant point spread function a reasonable imaging model?
- **Lost detail.** Which details become weakly measured after blur?
- **Stabilization.** What does the stabilizer trade away when it suppresses noise amplification?

## Checkpoint

Answer in your own words:

1. What does a normalized blur kernel preserve?
2. Why is a point spread function a useful model?
3. Which boundary mode would you choose for a natural photograph?
4. Why does naive inverse filtering amplify noise?